In [1]:
# Setup
import gradio as gr
import pandas as pd
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.output_parsers import PydanticOutputParser
from langchain.schema.output_parser import OutputParserException
from langchain_core.documents import Document
from pydantic import BaseModel


In [2]:

import os
from dotenv import load_dotenv

# TODO: Load environment variables and API Keys here
load_dotenv(override=True)
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ API KEY:")

In [3]:

# Load and preprocess data
loader = CSVLoader(file_path="jiomart_products_database.csv", source_column="title")
documents_raw = loader.load()


In [4]:

# Convert page_content string to dict and build metadata
documents = []
for doc in documents_raw:
    try:
        row_data = dict(
            line.split(":", 1) for line in doc.page_content.split("\n") if ":" in line
        )
        row_data = {k.strip(): v.strip() for k, v in row_data.items()}

        page_text = f"Name: {row_data.get('title', '')} | Sub-type: {row_data.get('subType', '')} | Type: {row_data.get('type', '')} | Price: {row_data.get('discountedPrice', 0)} | Image: {row_data.get('filename', '')}"
        metadata = {
            "category": row_data.get("type", ""),
            "sub_category": row_data.get("subType", "")
        }

        documents.append(Document(page_content=page_text, metadata=metadata))

    except Exception as e:
        print("Skipping row due to error:", e)


In [5]:
documents

[Document(metadata={'category': 'Staples', 'sub_category': 'Atta, Flours and Sooji'}, page_content='Name: Besan 1 kg | Sub-type: Atta, Flours and Sooji | Type: Staples | Price: 74 | Image: https://www.jiomart.com/images/product/150x150/491349649/besan-1-kg-0-20210521.jpg'),
 Document(metadata={'category': 'Staples', 'sub_category': 'Atta, Flours and Sooji'}, page_content='Name: Besan 500 g | Sub-type: Atta, Flours and Sooji | Type: Staples | Price: 37 | Image: https://www.jiomart.com/images/product/150x150/491349648/besan-500-g-0-20210521.jpg'),
 Document(metadata={'category': 'Staples', 'sub_category': 'Atta, Flours and Sooji'}, page_content='Name: Maida 1 kg | Sub-type: Atta, Flours and Sooji | Type: Staples | Price: 32 | Image: https://www.jiomart.com/images/product/150x150/491349662/maida-1-kg-0-20210521.jpg'),
 Document(metadata={'category': 'Staples', 'sub_category': 'Atta, Flours and Sooji'}, page_content='Name: Rawa 1 kg | Sub-type: Atta, Flours and Sooji | Type: Staples | Pric

In [22]:

# TODO: Create embeddings
embeddings_model = OllamaEmbeddings(
    model="nomic-embed-text"
)


In [23]:

# TODO: Store in Chroma
documents_for_chroma = documents[:100]


In [24]:
import gradio as gr
import pandas as pd
import os

from dotenv import load_dotenv

from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain.chat_models import init_chat_model
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.exceptions import OutputParserException

from pydantic import BaseModel
from typing import List


In [25]:

vectorstore = Chroma.from_documents(
    documents=documents_for_chroma,
    embedding=embeddings_model,
    collection_name="jiomart_products"
)


In [26]:

retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 5
    }
)



In [27]:

class GroceryItem(BaseModel):

    item_name: str

    price: float

    quantity: int

    image_url: str


class GroceryOutput(BaseModel):

    reasoning: str

    items: List[GroceryItem]

# TODO: Create Pydantic schema - Create two classes 1) GroceryItem with item_name, price, quantity, image_url and 2) GroceryOutput with reasoning and items. Pay attention to how you would create these classes

# Initialize parser
parser = PydanticOutputParser(pydantic_object=GroceryOutput)

# TODO: Prompt Template
prompt = ChatPromptTemplate.from_template(
    """
You are a smart grocery shopping assistant.

Your task is to create a grocery shopping cart based on
the user's preferences.

USER PREFERENCES:
{preferences}

AVAILABLE PRODUCTS:
{context}

IMPORTANT RULES:

1. Recommend products ONLY from the available products
   provided in the context.

2. Do NOT invent products.

3. Do NOT recommend products that the user explicitly
   excluded.

4. Use the exact product name from the context.

5. Use the exact price from the context.

6. Use the image URL from the context.

7. Select a reasonable quantity based on the user's request.

8. If the user asks for a specific type of product,
   prioritize products matching that requirement.

9. Keep the reasoning concise and explain why the selected
   products match the user's preferences.

10. Return ONLY the requested structured output.

{format_instructions}
"""
)


In [28]:

# TODO: Create a list of LLMs that will be used
# Use gpt-oss-120b and 2 other LLMs from Groq - make sure to check leaderboard to explore which ones to use

model_name_map = {
    "GPT-OSS-120B": "openai/gpt-oss-120b",
    "Llama 3.3 70B":
        "llama-3.3-70b-versatile",
    "Llama 4 Scout":
        "meta-llama/llama-4-scout-17b-16e-instruct"
}

model_choices = [
    "GPT-OSS-120B (Groq)",
    "Llama 3.3 70B (Groq)",
    "Llama 4 Scout (Groq)"
]
# TODO: Model selector
def get_llm(model_choice, temperature):

    # Remove "(Groq)" from UI name
    model_key = model_choice.replace(
        " (Groq)",
        ""
    )

    # Get actual Groq model ID
    model_name = model_name_map[model_key]

    # Initialize Groq chat model
    llm = init_chat_model(
        model=model_name,
        model_provider="groq",
        temperature=temperature
    )

    return llm
        


In [29]:
# TODO: RAG pipeline
def generate_cart(model_choice, user_input):

    try:

        # ----------------------------------------------------
        # STEP 1: Retrieve relevant documents
        # ----------------------------------------------------

        preferences = user_input["preferences"]

        context_docs = retriever.invoke(
            preferences
        )


        # ----------------------------------------------------
        # STEP 2: Convert retrieved documents to text
        # ----------------------------------------------------

        relevant_text = "\n\n".join(
            doc.page_content
            for doc in context_docs
        )


        # ----------------------------------------------------
        # STEP 3: Get selected LLM
        # ----------------------------------------------------

        llm = get_llm(
            model_choice,
            user_input["temperature"]
        )


        # ----------------------------------------------------
        # STEP 4: Format prompt
        # ----------------------------------------------------

        formatted_prompt = prompt.format(
            preferences=preferences,
            context=relevant_text,
            format_instructions=(
                parser.get_format_instructions()
            )
        )


        # ----------------------------------------------------
        # STEP 5: Call LLM
        # ----------------------------------------------------

        output = llm.invoke(
            formatted_prompt
        )


        # ----------------------------------------------------
        # STEP 6: Parse output
        # ----------------------------------------------------

        try:

            result = parser.parse(
                output.content
            )

        except OutputParserException:

            return (
                "Could not parse output.",
                None
            )


        return result


    except Exception as e:

        print(
            "Error in generate_cart:",
            e
        )

        return (
            f"Error: {str(e)}",
            None
        )


In [30]:
# def generate_cart(model_choice, user_input):
    # context_docs = 
    # relevant_text = 
    # llm = 
    # prompt = 
    # output =
    # try:
    #     result = parser.parse(output.content)
    # except OutputParserException:
    #     return "Could not parse output.", None
    # return result

# User Interface
def gradio_interface(preferences, model_choice, temperature):
    user_input = {
        "preferences": preferences,
        "model_choice": model_choice,
        "temperature": temperature,
    }
    result = generate_cart(model_choice, user_input)
    if not result or isinstance(result, str):
        return result, None
    if isinstance(result, tuple):
        explanation, _ = result
        return explanation, None

    # Build a gallery format: [(image_url, caption), ...]
    gallery_items = [
        (item.image_url, f"{item.item_name}\nQty.{item.quantity}\n₹{item.quantity*item.price}\n")
        for item in result.items
    ]
    return result.reasoning, gallery_items

demo = gr.Interface(
    fn=gradio_interface,
    inputs=[
        gr.Textbox(label="Describe your grocery needs (e.g., 'high protein, no besan or curd')"),
        gr.Dropdown(label="Model", choices=model_choices),
        gr.Slider(minimum=0.0, maximum=1.5, value=0.7, step=0.1, label="Temperature")
    ],
    outputs=[
        gr.Textbox(label="Considerations"),
        gr.Gallery(label="Shopping Cart", columns=3, height="auto")
    ],
    title="Smart Grocery Cart Assistant",
    description="Get a product list tailored to your dietary preferences."
)

# Launch UI
if __name__ == "__main__":
    demo.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
